# 03 - Modèles BLIP et GIT

Ce notebook est prévu pour Google Colab avec GPU. Il compare BLIP et GIT
sur les mêmes 500 images Flickr8k, en précision normale, 8 bits et 4 bits.
Les légendes générées et les mesures sont sauvegardées dans Google Drive.

Ce notebook ne doit pas être exécuté dans VS Code.

In [ ]:
# Installer les bibliothèques nécessaires dans Google Colab.
!pip -q install datasets transformers accelerate bitsandbytes

In [ ]:
# Connecter Google Drive et préparer les dossiers du projet.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/ia_visuelle")
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import gc
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import BitsAndBytesConfig

SEED = 42
N_IMAGES = 500
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError("Un GPU Colab est n?cessaire pour ex?cuter ce notebook.")

# V?rifier et r?utiliser le CSV produit par le notebook 01.
references_path = RESULTS_DIR / "flickr8k_references.csv"
if not references_path.exists():
    raise FileNotFoundError(
        f"CSV introuvable : {references_path}. Ex?cutez le notebook 01."
    )
references = pd.read_csv(references_path)
required_columns = ["id"] + [f"caption_{number}" for number in range(5)]
missing = [column for column in required_columns if column not in references]
if missing:
    raise ValueError(f"Colonnes manquantes dans le CSV : {missing}")

# Reprendre exactement les images du notebook 01.
dataset = load_dataset("jxie/flickr8k", split="train")
sample = dataset.shuffle(seed=SEED).select(range(min(N_IMAGES, len(dataset))))
if len(references) != len(sample):
    raise ValueError("Le nombre d images ne correspond pas au notebook 01.")
for number in range(5):
    expected = [example[f"caption_{number}"] for example in sample]
    actual = references[f"caption_{number}"].tolist()
    if actual != expected:
        raise ValueError(f"La s?lection caption_{number} ne correspond pas au notebook 01.")

print(f"Images utilis?es : {len(sample)}")
print(f"GPU utilis? : {torch.cuda.get_device_name(0)}")

## Fonctions communes

Les fonctions suivantes chargent un modèle, génèrent une légende par image,
mesurent le temps total et enregistrent les résultats. Une configuration
déjà enregistrée est ignorée pour permettre de reprendre après une coupure.

In [ ]:
def quantization_config(precision):
    """Créer la configuration bitsandbytes demandée."""
    if precision == "fp16":
        return None
    if precision == "int8":
        return BitsAndBytesConfig(load_in_8bit=True)
    if precision == "int4":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    raise ValueError(f"Précision inconnue : {precision}")


def clear_gpu():
    """Libérer le modèle précédent avant de charger le suivant."""
    gc.collect()
    torch.cuda.empty_cache()


def output_path(model_name, precision):
    return RESULTS_DIR / f"predictions_{model_name}_{precision}.csv"


In [ ]:
def run_blip(precision):
    """G?n?rer des l?gendes avec BLIP et mesurer temps/m?moire."""
    from transformers import BlipForConditionalGeneration, BlipProcessor

    model_id = "Salesforce/blip-image-captioning-base"
    processor = BlipProcessor.from_pretrained(model_id)
    model = BlipForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quantization_config(precision),
        torch_dtype=torch.float16 if precision == "fp16" else None,
        device_map="auto",
    )
    model.eval()
    torch.cuda.reset_peak_memory_stats()

    rows = []
    start = time.perf_counter()
    for image_id, example in enumerate(sample):
        inputs = processor(images=example["image"], return_tensors="pt")
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        with torch.no_grad():
            tokens = model.generate(**inputs, max_new_tokens=30)
        caption = processor.decode(tokens[0], skip_special_tokens=True)
        rows.append({"image_id": image_id, "caption": caption})

    elapsed = time.perf_counter() - start
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return rows, elapsed, peak_memory_mb

In [ ]:
def run_git(precision):
    """G?n?rer des l?gendes avec GIT et mesurer temps/m?moire."""
    from transformers import GitForCausalLM, GitProcessor

    model_id = "microsoft/git-base"
    processor = GitProcessor.from_pretrained(model_id)
    model = GitForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config(precision),
        torch_dtype=torch.float16 if precision == "fp16" else None,
        device_map="auto",
    )
    model.eval()
    torch.cuda.reset_peak_memory_stats()

    rows = []
    start = time.perf_counter()
    for image_id, example in enumerate(sample):
        inputs = processor(images=example["image"], return_tensors="pt")
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        with torch.no_grad():
            tokens = model.generate(**inputs, max_new_tokens=30)
        caption = processor.batch_decode(tokens, skip_special_tokens=True)[0]
        rows.append({"image_id": image_id, "caption": caption})

    elapsed = time.perf_counter() - start
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return rows, elapsed, peak_memory_mb

In [ ]:
experiments = {
    "blip": run_blip,
    "git": run_git,
}
precisions = ["fp16", "int8", "int4"]
summary_path = RESULTS_DIR / "model_inference_summary.csv"
summary = []
if summary_path.exists():
    summary = pd.read_csv(summary_path).to_dict("records")

for model_name, run_model in experiments.items():
    for precision in precisions:
        path = output_path(model_name, precision)
        if path.exists():
            print(f"D?j? pr?sent, ignor? : {path.name}")
            continue

        clear_gpu()
        print(f"D?marrage : {model_name} - {precision}")
        predictions, elapsed, peak_memory_mb = run_model(precision)
        pd.DataFrame(predictions).to_csv(path, index=False)
        summary.append({
            "model": model_name,
            "precision": precision,
            "seconds": elapsed,
            "images": len(predictions),
            "seconds_per_image": elapsed / len(predictions),
            "peak_memory_mb": peak_memory_mb,
        })
        pd.DataFrame(summary).to_csv(summary_path, index=False)
        print(f"Sauvegard? : {path}")

In [ ]:
# Sauvegarder le résumé des temps d'inférence.
summary_path = RESULTS_DIR / "model_inference_summary.csv"
if summary:
    pd.DataFrame(summary).to_csv(summary_path, index=False)
    print(f"Résumé sauvegardé : {summary_path}")
elif summary_path.exists():
    print(f"Résumé déjà présent : {summary_path}")
else:
    raise FileNotFoundError("Aucune expérience n'a été exécutée.")

## Vérification

Les fichiers `predictions_blip_*.csv`, `predictions_git_*.csv` et
`model_inference_summary.csv` seront utilisés par le notebook 04.
Les images et les modèles téléchargés restent dans Colab/Google Drive et
ne doivent pas être ajoutés au dépôt GitHub.